In [1]:
import os 
import pandas as pd 
import logging 
import traceback
import numpy as np 
from basicprocess import *

In [2]:
'''Setup & Main Execution'''
# 00_Setup 所有全域函數
logfile = os.path.abspath(os.path.join(os.getcwd(), '..', 'Log', '台鐵分析.log'))
# if os.path.exists(logfile):
#     os.remove(logfile)
    
logging.basicConfig(
    filename=logfile,
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)


In [ ]:
logging.info("Job Started")

# settup 
originalticket_folder = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '2024_2025'))
TRA_folder = os.path.join(originalticket_folder,'臺鐵電子票證資料(TO1A)') #臺鐵電子票證資料(TO1A)

# 初步篩選整理票證
TRA_primary_organized_folder = os.path.join(os.getcwd(), '..', '06_台鐵','01_初步篩選整理票證')
TRA_timeselectfolder = create_folder(os.path.join(TRA_primary_organized_folder, '01_指定時間區間票證'))


In [ ]:
def TRA_tikcets_read_and_format(df, starttime, endtime):
    df['Infodate'] = pd.to_datetime(df['Infodate'], format='%Y-%m-%d',errors = 'coerce')
    df['EntryTime'] = pd.to_datetime(df['EntryTime'], format='%Y-%m-%d %H:%M:%S', errors = 'coerce')
    df['ExitTime'] = pd.to_datetime(df['ExitTime'], format='%Y-%m-%d %H:%M:%S', errors = 'coerce')
    df['Hour'] = df['EntryTime'].dt.hour
    df['BordingHour'] = df['ExitTime'].dt.hour
    df['DeboardingHour'] = df['EntryTime'].dt.hour
    df['EntryStationID'] = df['EntryStationID'].astype('int64')
    df['ExitStationID'] = df['ExitStationID'].astype('int64')
    df['TimeSelect'] = (df['Infodate'] >= starttime) & (df['Infodate'] <= endtime)
    df['Error'] = df['EntryTime'] > df['ExitTime']

    return df

# def step1_selectedtime(filepath, chunksize, starttime, endtime, outputfolder):

#     logging.info("開始讀取原始檔案並且篩選時間區間")
#     logging.info(f"指定篩選時間為:{starttime}~{endtime}")


#     outputpath = os.path.join(outputfolder, os.path.basename(filepath).replace('.csv', f'_{starttime}_to_{endtime}.csv'))
#     if os.path.exists(outputpath):
#         logging.warning(f'{os.path.basename(outputpath)} 已經存在，不進行讀取輸出；請再次檢查。')

#     outputpath2 = os.path.join(outputfolder, os.path.basename(filepath).replace('.csv', '_合理資料.csv'))
#     if os.path.exists(outputpath):
#         logging.warning(f'{os.path.basename(outputpath2)} 已經存在，不進行讀取輸出；請再次檢查。')

#     chunks = pd.read_csv(filepath, skiprows=1, chunksize=chunksize)
#     first_chunk = True

#     for chunk in chunks:
#         chunk = TRA_tikcets_read_and_format(df = chunk, starttime = starttime, endtime = endtime)
#         chunk = chunk[(chunk['TimeSelect'] == True)].drop(columns = ['TimeSelect'])
#         if len(chunk) > 0 : 
#             chunk.to_csv(
#                 outputpath,
#                 mode='w' if first_chunk else 'a',
#                 header=first_chunk,
#                 index=False,
#                 encoding='utf-8-sig'
#             )
#             chunk = chunk[(chunk['Error'] == False)].drop(columns = ['Error'])
#             if len(chunk) > 0 : 
#                 chunk.to_csv(
#                     outputpath2,
#                     mode='w' if first_chunk else 'a',
#                     header=first_chunk,
#                     index=False,
#                     encoding='utf-8-sig'
#                 )

#         first_chunk = False

#     logging.info(f"篩選出時間為 {starttime}~{endtime} 的資料")

def step1_selectedtime(filepath, chunksize, starttime, endtime, outputfolder):
    logging.info("開始讀取原始檔案並且篩選時間區間")
    logging.info(f"指定篩選時間為:{starttime}~{endtime}")

    outputpath = os.path.join(
        outputfolder,
        os.path.basename(filepath).replace('.csv', f'_{starttime}_to_{endtime}.csv')
    )
    outputpath2 = os.path.join(
        outputfolder,
        os.path.basename(filepath).replace('.csv', '_合理資料.csv')
    )

    if os.path.exists(outputpath):
        logging.warning(f'{os.path.basename(outputpath)} 已經存在，不進行讀取輸出；請再次檢查。')
        return
    if os.path.exists(outputpath2):  # <- 你原本這裡也在檢查 outputpath（寫錯了）
        logging.warning(f'{os.path.basename(outputpath2)} 已經存在，不進行讀取輸出；請再次檢查。')
        return

    chunks = pd.read_csv(filepath, skiprows=1, chunksize=chunksize)

    wrote_header_1 = False
    wrote_header_2 = False

    for chunk in chunks:
        chunk = TRA_tikcets_read_and_format(df=chunk, starttime=starttime, endtime=endtime)

        # --- 檔案 1：時間區間資料 ---
        chunk_time = chunk[chunk['TimeSelect']].drop(columns=['TimeSelect'])
        if len(chunk_time) > 0:
            chunk_time.to_csv(
                outputpath,
                mode='w' if not wrote_header_1 else 'a',
                header=not wrote_header_1,
                index=False,
                encoding='utf-8-sig'
            )
            wrote_header_1 = True

        # --- 檔案 2：合理資料（Error=False）---
        chunk_ok = chunk_time[chunk_time['Error'] == False].drop(columns=['Error'])
        if len(chunk_ok) > 0:
            chunk_ok.to_csv(
                outputpath2,
                mode='w' if not wrote_header_2 else 'a',
                header=not wrote_header_2,
                index=False,
                encoding='utf-8-sig'
            )
            wrote_header_2 = True

    logging.info(f"篩選出時間為 {starttime}~{endtime} 的資料")


def main():
    # 讀取台鐵票證資料 篩選正確區間
    step1_selectedtime(filepath = findfiles(TRA_folder)[0], 
                       chunksize = 10000, 
                       starttime = '2024-10-01', 
                       endtime = '2024-11-30', 
                       outputfolder = TRA_timeselectfolder)


In [ ]:
# logging.info("開始進行統計筆數")

files = findfiles(TRA_timeselectfolder)
files =  [f for f in files if '合理' in f]
# df = read_combined_dataframe(files)
df = pd.read_csv(files[0], nrows = 1000)

groupbycolumns =['HolderType', 'EntryStationID', 'EntryStationName', 'ExitStationID', 'ExitStationName',  'Infodate', 'Hour', 'BordingHour', 'DeboardingHour']
df.reindex(columns=groupbycolumns).groupby(groupbycolumns).size().reset_index(name='Count')
df['DaysofWeek'] = df['Infodate'].dt.dayofweek
wdwk_1_condition = df['DaysofWeek'].isin([1, 2, 3])
# WDWK = -1 (週六=5, 週日=6)
wdwk_neg1_condition = df['DaysofWeek'].isin([5, 6])
# 使用 np.select (比多個 if/elif 判斷更快)
df['WDWK'] = np.select(
    [wdwk_1_condition, wdwk_neg1_condition], # 條件列表
    [1, 0],                                # 對應的值
    default=-1                               # 預設值 (其他日子=1)
)


In [ ]:
df = pd.read_csv(r"D:\B-Project\2025\6800\Technical\12票證資料\TicketAnalysis\06_台鐵\01_初步篩選整理票證\01_指定時間區間票證\臺鐵電子票證資料(TO1A)_2024-10-01_to_2024-11-30.csv", 
                 nrows = 500)
df.columns

In [ ]:
df.head()

In [ ]:
if __name__ == "__main__":
    try:
        main()

    except Exception as e:
        logging.error("main() 執行失敗：%s", e)  
        logging.error("Traceback:\n%s", traceback.format_exc())
    
    outputlog(logfile=logfile)